<a href="https://colab.research.google.com/github/umutbarandemir/CampusCam/blob/main/Kamp%C3%BCsFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# HÜCRE 1 — KURULUM
# ============================================================
import subprocess, sys

print("📦 Gerekli kütüphaneler kuruluyor... (1-2 dakika sürebilir)")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics>=8.0.0",
    "gradio>=4.0.0",
    "opencv-python-headless",
    "pandas",
    "matplotlib",
    "Pillow"
], check=True)
print("✅ Kurulum tamamlandı!")


# %%

📦 Gerekli kütüphaneler kuruluyor... (1-2 dakika sürebilir)
✅ Kurulum tamamlandı!


In [ ]:
# ============================================================
# HÜCRE 2 — KÜTÜPHANELERİ İÇE AKTAR VE CİHAZ KONTROL
# ============================================================
import torch
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights
)
from ultralytics import YOLO

import cv2
import numpy as np
import time
import pandas as pd
import gradio as gr
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

# Cihaz seçimi (GPU varsa GPU, yoksa CPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Kullanılan cihaz: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  GPU bulunamadı. CPU kullanılıyor — Faster R-CNN yavaş çalışacak.")
    print("   Colab'da: Çalışma Zamanı → Çalışma Zamanı Türünü Değiştir → T4 GPU")

print(f"\n📦 PyTorch: {torch.__version__}")
print(f"📦 Torchvision: {torchvision.__version__}")



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🖥️  Kullanılan cihaz: cpu
   ⚠️  GPU bulunamadı. CPU kullanılıyor — Faster R-CNN yavaş çalışacak.
   Colab'da: Çalışma Zamanı → Çalışma Zamanı Türünü Değiştir → T4 GPU

📦 PyTorch: 2.11.0+cpu
📦 Torchvision: 0.26.0+cpu


In [ ]:
# ============================================================
# HÜCRE 3 — COCO SINIF İSİMLERİ (80 sınıf)
# ============================================================
# Faster R-CNN 1-indexed kullanır: 0 = background, 1 = person, ...
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack',
    'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple',
    'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
    'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote',
    'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink',
    'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush'
]

# Renk paleti (RGB formatı — Gradio için)
YOLO_COLOR_RGB  = (50, 205, 50)    # Lime Green
FRCNN_COLOR_RGB = (255, 140, 0)    # Dark Orange

# Renk paleti (BGR formatı — OpenCV için)
YOLO_COLOR_BGR  = (50, 205, 50)[::-1]
FRCNN_COLOR_BGR = (255, 140, 0)[::-1]

print(f"✅ {len(COCO_CLASSES) - 1} COCO sınıfı yüklendi")
print(f"   Kampüs için önemli sınıflar: person, bicycle, car, motorcycle, bus, backpack")

✅ 80 COCO sınıfı yüklendi
   Kampüs için önemli sınıflar: person, bicycle, car, motorcycle, bus, backpack


In [ ]:
# ============================================================
# HÜCRE 4 — MODELLERİ YÜKLE
# ============================================================

# ---- YOLOv8n ----
print("⏳ YOLOv8n yükleniyor...")
yolo_model = YOLO('yolov8n.pt')   # 'n' = nano, en hızlı versiyon
yolo_model.to(DEVICE)
print("✅ YOLOv8n hazır!")

# ---- Faster R-CNN (ResNet-50 + FPN) ----
print("⏳ Faster R-CNN (ResNet-50-FPN) yükleniyor... (daha büyük model, biraz sürer)")
frcnn_weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
frcnn_model = fasterrcnn_resnet50_fpn(weights=frcnn_weights)
frcnn_model.to(DEVICE)
frcnn_model.eval()
frcnn_transforms = frcnn_weights.transforms()
print("✅ Faster R-CNN hazır!")

# ---- Model bilgisi ----
yolo_params  = sum(p.numel() for p in yolo_model.model.parameters())
frcnn_params = sum(p.numel() for p in frcnn_model.parameters())
print(f"\n📊 Model karşılaştırması:")
print(f"   YOLOv8n     : {yolo_params/1e6:.1f}M parametre | Single-stage | Anchor-free")
print(f"   Faster R-CNN: {frcnn_params/1e6:.1f}M parametre | Two-stage   | Anchor-based")


⏳ YOLOv8n yükleniyor...
✅ YOLOv8n hazır!
⏳ Faster R-CNN (ResNet-50-FPN) yükleniyor... (daha büyük model, biraz sürer)
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:05<00:00, 29.3MB/s]


✅ Faster R-CNN hazır!

📊 Model karşılaştırması:
   YOLOv8n     : 3.2M parametre | Single-stage | Anchor-free
   Faster R-CNN: 41.8M parametre | Two-stage   | Anchor-based


In [ ]:
# ============================================================
# HÜCRE 5 — DETECTION FONKSİYONLARI
# ============================================================

def detect_yolo(image_rgb, conf_threshold=0.50, iou_threshold=0.45):
    """
    YOLOv8n ile nesne tespiti.

    Mimari notu:
    - Single-stage: backbone → neck (PAN) → head (tek geçiş)
    - Anchor-free: her grid cell için offset tahmin eder
    - NMS (Non-Maximum Suppression) iou_threshold parametresiyle kontrol edilir

    Args:
        image_rgb (np.ndarray): RGB formatında görüntü [H,W,3]
        conf_threshold (float): Minimum güven skoru (0.0–1.0)
        iou_threshold  (float): NMS IoU eşiği (0.0–1.0)

    Returns:
        boxes      : [[x1,y1,x2,y2], ...] piksel koordinatları
        scores     : [float, ...] güven skorları
        class_ids  : [int, ...]
        class_names: [str, ...]
        time_ms    : çıkarım süresi (ms)
    """
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    results = yolo_model(
        image_rgb,
        conf=conf_threshold,
        iou=iou_threshold,
        verbose=False
    )

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    time_ms = (time.perf_counter() - t0) * 1000

    boxes, scores, class_ids, class_names = [], [], [], []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            score   = float(box.conf[0].cpu())
            cls_id  = int(box.cls[0].cpu())
            cls_name = yolo_model.names[cls_id]
            boxes.append([x1, y1, x2, y2])
            scores.append(score)
            class_ids.append(cls_id)
            class_names.append(cls_name)

    return boxes, scores, class_ids, class_names, time_ms


def detect_frcnn(image_rgb, conf_threshold=0.50):
    """
    Faster R-CNN ile nesne tespiti.

    Mimari notu:
    - Stage 1 (RPN): Region Proposal Network — olası nesne bölgelerini üretir
    - Stage 2 (Head): Her bölge için sınıflandırma + kutu düzeltmesi yapar
    - FPN: Feature Pyramid Network — çok ölçekli özellik haritaları

    Neden iki aşama?
    - RPN object/background ayrımı yaparak yüksek recall'u yakalar
    - Head ise bu bölgelerde ince sınıflandırma yapar → daha yüksek precision

    Args:
        image_rgb     (np.ndarray): RGB formatında görüntü [H,W,3]
        conf_threshold (float)    : Minimum güven skoru

    Returns:
        boxes, scores, class_ids, class_names, time_ms
    """
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()

    pil_img = Image.fromarray(image_rgb)
    img_tensor = frcnn_transforms(pil_img).to(DEVICE)

    with torch.no_grad():
        predictions = frcnn_model([img_tensor])

    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()
    time_ms = (time.perf_counter() - t0) * 1000

    pred = predictions[0]
    keep = pred['scores'] > conf_threshold

    raw_boxes  = pred['boxes'][keep].cpu().numpy()
    raw_scores = pred['scores'][keep].cpu().numpy()
    raw_labels = pred['labels'][keep].cpu().numpy().astype(int)

    boxes, scores, class_ids, class_names = [], [], [], []
    for box, score, label in zip(raw_boxes, raw_scores, raw_labels):
        if label >= len(COCO_CLASSES):
            continue
        cls_name = COCO_CLASSES[label]
        if cls_name in ('__background__', 'N/A'):
            continue
        boxes.append(list(map(int, box)))
        scores.append(float(score))
        class_ids.append(int(label))
        class_names.append(cls_name)

    return boxes, scores, class_ids, class_names, time_ms


def draw_detections(image_rgb, boxes, scores, class_names, color_rgb, model_label):
    """
    Görüntü üzerine bounding box + etiket çiz.
    Hem YOLOv8 hem Faster R-CNN sonuçları için kullanılır.

    Args:
        image_rgb   : RGB numpy array
        boxes       : [[x1,y1,x2,y2], ...]
        scores      : [float, ...]
        class_names : [str, ...]
        color_rgb   : (R, G, B) tuple
        model_label : Görüntü başlığı için model adı

    Returns:
        Annotated görüntü (RGB numpy array)
    """
    img = image_rgb.copy()
    h, w = img.shape[:2]

    # Görüntü boyutuna göre dinamik font/kalınlık
    font_scale = max(0.40, min(0.90, w / 1200))
    thickness  = max(1, int(w / 600))
    color_bgr  = color_rgb[::-1]   # PIL/matplotlib RGB → OpenCV BGR

    # OpenCV BGR'ye çevir (draw için)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

    for box, score, cls_name in zip(boxes, scores, class_names):
        x1, y1, x2, y2 = box

        # Bounding box
        cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color_bgr, thickness + 1)

        # Etiket
        label_text = f"{cls_name}: {score:.2f}"
        (tw, th), baseline = cv2.getTextSize(
            label_text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness
        )
        label_y = max(y1 - 4, th + baseline + 4)

        # Etiket arka planı (okunabilirlik için)
        cv2.rectangle(
            img_bgr,
            (x1, label_y - th - baseline - 2),
            (x1 + tw + 6, label_y + baseline),
            color_bgr, -1
        )
        # Etiket yazısı (renk parlaklığına göre siyah/beyaz)
        brightness = 0.299*color_rgb[0] + 0.587*color_rgb[1] + 0.114*color_rgb[2]
        text_color = (0, 0, 0) if brightness > 128 else (255, 255, 255)
        cv2.putText(
            img_bgr, label_text, (x1 + 3, label_y - 2),
            cv2.FONT_HERSHEY_SIMPLEX, font_scale, text_color, thickness, cv2.LINE_AA
        )

    # Model başlığı (sol üst köşe)
    header = f"{model_label}  |  {len(boxes)} nesne"
    cv2.putText(
        img_bgr, header, (10, 32),
        cv2.FONT_HERSHEY_SIMPLEX, 0.8, color_bgr, 2, cv2.LINE_AA
    )

    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)



In [ ]:
# ============================================================
# HÜCRE 6 — KARŞILAŞTIRMA VE METRİK FONKSİYONLARI
# ============================================================

def compare_models(input_image, conf_threshold=0.50, iou_threshold=0.45):
    """
    Her iki modeli aynı görüntü üzerinde çalıştırır ve karşılaştırır.
    Gradio arayüzü tarafından çağrılır.

    Returns:
        yolo_img   : YOLOv8 sonucu (RGB numpy)
        frcnn_img  : Faster R-CNN sonucu (RGB numpy)
        metrics_df : Karşılaştırma tablosu (pandas DataFrame)
    """
    if input_image is None:
        return None, None, pd.DataFrame({"Uyarı": ["Lütfen bir görüntü yükleyin."]})

    # Gradio numpy array verir (RGB)
    image_rgb = input_image if isinstance(input_image, np.ndarray) else np.array(input_image)

    # ── YOLOv8 ──────────────────────────────────────────────
    yolo_boxes, yolo_scores, yolo_ids, yolo_names, yolo_ms = detect_yolo(
        image_rgb, conf_threshold, iou_threshold
    )
    yolo_img = draw_detections(
        image_rgb, yolo_boxes, yolo_scores, yolo_names,
        YOLO_COLOR_RGB, "YOLOv8n"
    )

    # ── Faster R-CNN ─────────────────────────────────────────
    frcnn_boxes, frcnn_scores, frcnn_ids, frcnn_names, frcnn_ms = detect_frcnn(
        image_rgb, conf_threshold
    )
    frcnn_img = draw_detections(
        image_rgb, frcnn_boxes, frcnn_scores, frcnn_names,
        FRCNN_COLOR_RGB, "Faster R-CNN"
    )

    # ── Metrik Tablosu ───────────────────────────────────────
    # Sınıf bazlı sayım
    def count_by_class(names):
        c = {}
        for n in names:
            c[n] = c.get(n, 0) + 1
        return c

    yolo_cc  = count_by_class(yolo_names)
    frcnn_cc = count_by_class(frcnn_names)
    all_cls  = sorted(set(list(yolo_cc) + list(frcnn_cc)))

    rows = [
        {"Metrik": "⚡ Toplam Tespit",
         "YOLOv8n": len(yolo_boxes), "Faster R-CNN": len(frcnn_boxes)},
        {"Metrik": "⏱️ Çıkarım Süresi (ms)",
         "YOLOv8n": f"{yolo_ms:.1f}", "Faster R-CNN": f"{frcnn_ms:.1f}"},
        {"Metrik": "🚀 Anlık FPS",
         "YOLOv8n": f"{1000/yolo_ms:.1f}" if yolo_ms > 0 else "—",
         "Faster R-CNN": f"{1000/frcnn_ms:.1f}" if frcnn_ms > 0 else "—"},
        {"Metrik": "📊 Ort. Güven Skoru",
         "YOLOv8n": f"{np.mean(yolo_scores):.3f}" if yolo_scores else "—",
         "Faster R-CNN": f"{np.mean(frcnn_scores):.3f}" if frcnn_scores else "—"},
        {"Metrik": "─── Sınıf Bazlı ───",
         "YOLOv8n": "", "Faster R-CNN": ""},
    ]
    for cls in all_cls:
        rows.append({
            "Metrik": f"  {cls}",
            "YOLOv8n": yolo_cc.get(cls, 0),
            "Faster R-CNN": frcnn_cc.get(cls, 0)
        })

    return yolo_img, frcnn_img, pd.DataFrame(rows)


def fps_benchmark(input_image, n_runs=20):
    """
    N çalıştırma üzerinden ortalama FPS ölçer.

    Neden 20 çalıştırma?
    - İlk çalıştırma her zaman daha yavaştır (JIT derleme, cache ısınma)
    - 20 çalıştırma standart sapmayı düşürür, güvenilir ortalama verir
    - 'Warmup' çalıştırmaları istatistiğe dahil edilmez
    """
    if input_image is None:
        return pd.DataFrame({"Uyarı": ["Lütfen bir görüntü yükleyin."]})

    image_rgb = input_image if isinstance(input_image, np.ndarray) else np.array(input_image)
    print(f"🔄 Benchmark başladı ({n_runs} çalıştırma)...")

    # Isınma çalıştırmaları (sayılmaz)
    detect_yolo(image_rgb)
    detect_frcnn(image_rgb)

    yolo_times, frcnn_times = [], []

    for i in range(n_runs):
        _, _, _, _, yt = detect_yolo(image_rgb)
        _, _, _, _, ft = detect_frcnn(image_rgb)
        yolo_times.append(yt)
        frcnn_times.append(ft)
        if (i + 1) % 5 == 0:
            print(f"   {i+1}/{n_runs} tamamlandı...")

    results = {
        "Model":              ["YOLOv8n", "Faster R-CNN"],
        "Ort. Süre (ms)":    [f"{np.mean(yolo_times):.2f}", f"{np.mean(frcnn_times):.2f}"],
        "Std Sapma (ms)":    [f"{np.std(yolo_times):.2f}",  f"{np.std(frcnn_times):.2f}"],
        "Min (ms)":          [f"{np.min(yolo_times):.2f}",  f"{np.min(frcnn_times):.2f}"],
        "Max (ms)":          [f"{np.max(yolo_times):.2f}",  f"{np.max(frcnn_times):.2f}"],
        "Ortalama FPS":      [f"{1000/np.mean(yolo_times):.1f}", f"{1000/np.mean(frcnn_times):.1f}"],
        "Hız Oranı (YOLO/FRCNN)": [
            f"{np.mean(frcnn_times)/np.mean(yolo_times):.1f}x daha hızlı", "—"
        ],
    }
    df = pd.DataFrame(results)
    print("\n📊 Benchmark Sonuçları:")
    print(df.to_string(index=False))
    return df




In [ ]:
# ============================================================
# HÜCRE 7 — VİDEO İŞLEME FONKSİYONU
# ============================================================

def process_video(video_path, conf_threshold=0.50, iou_threshold=0.45,
                  frame_skip=1, max_seconds=20):
    """
    Video üzerinde her iki modeli kare kare çalıştırır ve annotated video üretir.

    Teknik notlar:
    - frame_skip=1 → her kare işlenir (en yavaş, en doğru)
    - frame_skip=2 → her 2. kare (2x hızlı, görsel akıcılık yeterli)
    - max_seconds   → Colab oturumu zaman aşımına karşı koruma
    - FFmpeg yeniden kodlama → tarayıcı uyumlu H.264/mp4 çıktısı

    Args:
        video_path    : Gradio'nun geçici dosya yolu (str)
        conf_threshold: Güven eşiği (0–1)
        iou_threshold : IoU eşiği — NMS için (0–1)
        frame_skip    : Kaç karede bir işlem yapılacak (1–5)
        max_seconds   : İşlenecek maks. video süresi (saniye)

    Returns:
        out_yolo  : YOLOv8 annotated video yolu
        out_frcnn : Faster R-CNN annotated video yolu
        df        : Kare bazlı istatistik tablosu
    """
    if video_path is None:
        return None, None, pd.DataFrame({"Uyarı": ["Video dosyası yükleyin."]})

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, None, pd.DataFrame({"Uyarı": ["Video açılamadı — farklı format deneyin."]})

    src_fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    max_frames = int(src_fps * max_seconds)
    out_fps    = max(1.0, src_fps / frame_skip)

    print(f"🎬 Video: {width}×{height} @ {src_fps:.0f} FPS")
    print(f"   Ayarlar: her {frame_skip}. kare işlenecek | maks {max_seconds}s → ~{max_frames} kare")

    # Geçici ham video dosyaları (mp4v codec)
    tmp_y  = "/tmp/yolo_raw.mp4"
    tmp_f  = "/tmp/frcnn_raw.mp4"
    out_y  = "/tmp/yolo_out.mp4"
    out_f  = "/tmp/frcnn_out.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    w_yolo  = cv2.VideoWriter(tmp_y, fourcc, out_fps, (width, height))
    w_frcnn = cv2.VideoWriter(tmp_f, fourcc, out_fps, (width, height))

    yolo_times,  frcnn_times  = [], []
    yolo_counts, frcnn_counts = [], []
    frame_idx = 0
    written   = 0

    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= max_frames:
            break

        # Kare atlama (frame_skip > 1 ise ara kareleri oku ama işleme)
        if frame_idx % frame_skip != 0:
            frame_idx += 1
            continue

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # ── YOLOv8 ──────────────────────────────────────────
        yb, ys, _, yn, yms = detect_yolo(frame_rgb, conf_threshold, iou_threshold)
        yolo_ann = draw_detections(
            frame_rgb, yb, ys, yn, YOLO_COLOR_RGB,
            f"YOLOv8n | kare {frame_idx} | {len(yb)} nesne"
        )
        w_yolo.write(cv2.cvtColor(yolo_ann, cv2.COLOR_RGB2BGR))
        yolo_times.append(yms)
        yolo_counts.append(len(yb))

        # ── Faster R-CNN ────────────────────────────────────
        fb, fs, _, fn, fms = detect_frcnn(frame_rgb, conf_threshold)
        frcnn_ann = draw_detections(
            frame_rgb, fb, fs, fn, FRCNN_COLOR_RGB,
            f"Faster R-CNN | kare {frame_idx} | {len(fb)} nesne"
        )
        w_frcnn.write(cv2.cvtColor(frcnn_ann, cv2.COLOR_RGB2BGR))
        frcnn_times.append(fms)
        frcnn_counts.append(len(fb))

        written   += 1
        frame_idx += 1

        if written % 15 == 0:
            elapsed_s = frame_idx / src_fps
            print(f"   ⏳ {written} kare işlendi ({elapsed_s:.1f}s / {max_seconds}s)...")

    cap.release()
    w_yolo.release()
    w_frcnn.release()
    print(f"✅ Toplam {written} kare işlendi. Video yeniden kodlanıyor...")

    # FFmpeg ile H.264 yeniden kodlama (tarayıcı uyumluluğu için zorunlu)
    os.system(f"ffmpeg -y -i {tmp_y} -c:v libx264 -pix_fmt yuv420p -movflags +faststart {out_y} -loglevel quiet")
    os.system(f"ffmpeg -y -i {tmp_f} -c:v libx264 -pix_fmt yuv420p -movflags +faststart {out_f} -loglevel quiet")
    print("✅ Video hazır!")

    # ── Metrik Tablosu ──────────────────────────────────────
    if not yolo_times:
        return None, None, pd.DataFrame({"Uyarı": ["Hiç kare işlenemedi."]})

    rows = [
        {"Metrik": "🎞️ İşlenen Kare",
         "YOLOv8n": written, "Faster R-CNN": written},
        {"Metrik": "⏱️ Ort. Çıkarım Süresi (ms)",
         "YOLOv8n": f"{np.mean(yolo_times):.1f}",
         "Faster R-CNN": f"{np.mean(frcnn_times):.1f}"},
        {"Metrik": "🚀 Ort. FPS (çıkarım hızı)",
         "YOLOv8n": f"{1000/np.mean(yolo_times):.1f}",
         "Faster R-CNN": f"{1000/np.mean(frcnn_times):.1f}"},
        {"Metrik": "📦 Ort. Tespit / Kare",
         "YOLOv8n": f"{np.mean(yolo_counts):.1f}",
         "Faster R-CNN": f"{np.mean(frcnn_counts):.1f}"},
        {"Metrik": "📦 Max Tespit (tek karede)",
         "YOLOv8n": max(yolo_counts),
         "Faster R-CNN": max(frcnn_counts)},
        {"Metrik": "📦 Min Tespit (en boş kare)",
         "YOLOv8n": min(yolo_counts),
         "Faster R-CNN": min(frcnn_counts)},
        {"Metrik": "⚡ Hız Avantajı",
         "YOLOv8n": f"{np.mean(frcnn_times)/np.mean(yolo_times):.1f}× daha hızlı",
         "Faster R-CNN": "—"},
        {"Metrik": "📹 Video Süresi (işlenen)",
         "YOLOv8n": f"{written * frame_skip / src_fps:.1f}s",
         "Faster R-CNN": f"{written * frame_skip / src_fps:.1f}s"},
    ]
    return out_y, out_f, pd.DataFrame(rows)


In [ ]:
# ============================================================
# HÜCRE 8 — GRADIO KULLANICI ARAYÜZÜ (video + webcam destekli)
# ============================================================

def build_and_launch():
    """
    Gradio tabanlı karşılaştırma arayüzü — 5 sekme:
      1. Görüntü Karşılaştırma  — fotoğraf + webcam
      2. Video Analizi          — video dosyası yükleme (YENİ)
      3. Failure Case Analizi   — başarısız durum inceleme
      4. FPS Benchmark          — hız ölçümü
      5. Model Bilgisi          — teknik detaylar
    """
    with gr.Blocks(title="Kampüs Güvenlik Sistemi", theme=gr.themes.Soft()) as demo:

        gr.Markdown("""
        # 🎓 Kampüs Güvenlik Kamerası — Object Detection Karşılaştırma Sistemi
        **OMÜ Yüksek Lisans | Bilgisayarlı Görü Dersi Final Projesi**
        **YOLOv8n** (yeşil) vs **Faster R-CNN** (turuncu) — COCO pre-trained ağırlıkları
        """)

        # ────────────────────────────────────────────────────────
        # SEKME 1: Görüntü Karşılaştırma (webcam desteği eklendi)
        # ────────────────────────────────────────────────────────
        with gr.Tab("📷 Görüntü / Webcam"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### ⚙️ Giriş & Ayarlar")
                    input_img = gr.Image(
                        label="Görüntü Yükle veya Kamera Kullan 📸",
                        type="numpy",
                        sources=["upload", "webcam"],   # ← webcam desteği
                        height=280
                    )
                    conf_slider = gr.Slider(
                        minimum=0.10, maximum=0.90, value=0.50, step=0.05,
                        label="Güven Eşiği (Confidence Threshold)"
                    )
                    iou_slider = gr.Slider(
                        minimum=0.10, maximum=0.90, value=0.45, step=0.05,
                        label="IoU Eşiği — NMS ayarı"
                    )
                    run_btn = gr.Button(
                        "🔍 Her İki Modeli Çalıştır",
                        variant="primary", size="lg"
                    )
                    gr.Markdown("""
                    **💡 Parametre rehberi:**
                    - Gündüz / net görüntü → güven eşiği: 0.45–0.55
                    - Kalabalık sahne → IoU: 0.35–0.45 (NMS daha toleranslı)
                    - Gece / düşük ışık → güven eşiği: 0.25–0.35
                    - Uzak mesafe nesneler → güven eşiği: 0.20–0.30
                    - 📷 Webcam: soldaki görüntü kutusunda kamera ikonuna tıkla
                    """)

                with gr.Column(scale=2):
                    gr.Markdown("### 📊 Tespit Sonuçları")
                    with gr.Row():
                        yolo_out  = gr.Image(label="⚡ YOLOv8n — Yeşil kutular",   height=320)
                        frcnn_out = gr.Image(label="🎯 Faster R-CNN — Turuncu kutular", height=320)
                    metrics_tbl = gr.DataFrame(
                        label="📈 Karşılaştırma Metrikleri",
                        wrap=True
                    )

            run_btn.click(
                fn=compare_models,
                inputs=[input_img, conf_slider, iou_slider],
                outputs=[yolo_out, frcnn_out, metrics_tbl]
            )

        # ────────────────────────────────────────────────────────
        # SEKME 2: Video Analizi (YENİ)
        # ────────────────────────────────────────────────────────
        with gr.Tab("🎬 Video Analizi"):
            gr.Markdown("""
            ## Video Üzerinde Karşılaştırmalı Nesne Tespiti
            Video dosyası yükleyin — her kare YOLOv8 ve Faster R-CNN ile işlenir,
            annotated video ve istatistik tablosu üretilir.

            > ⚠️ **Süre Tahmini:** GPU'da her kare ~100–150ms (her iki model birlikte).
            > 15 saniyelik 25fps video = ~375 kare → yaklaşık 1–2 dakika.
            > Uzun videolarda **Kare Atlama ≥ 2** veya **Maks. Süre = 10s** öneririz.
            """)

            with gr.Row():
                # Sol: giriş + ayarlar
                with gr.Column(scale=1):
                    video_input = gr.Video(
                        label="📹 Video Yükle (.mp4, .avi, .mov, .mkv)",
                        sources=["upload"]
                    )
                    v_conf = gr.Slider(0.10, 0.90, value=0.45, step=0.05,
                                       label="Güven Eşiği")
                    v_iou  = gr.Slider(0.10, 0.90, value=0.45, step=0.05,
                                       label="IoU Eşiği (NMS)")
                    v_skip = gr.Slider(1, 5, value=2, step=1,
                                       label="Kare Atlama — 1=her kare, 2=her 2. kare (2× hızlı)")
                    v_maxs = gr.Slider(5, 60, value=15, step=5,
                                       label="Maks. İşlenecek Süre (saniye)")
                    video_btn = gr.Button("🎬 Videoyu İşle", variant="primary", size="lg")

                    gr.Markdown("""
                    **💡 İpuçları:**
                    - Kare Atlama=1 → yüksek kalite, yavaş
                    - Kare Atlama=2 → 2x hızlı, kalite yeterli (önerilen)
                    - Kare Atlama=3 → 3x hızlı, hafif titreme
                    - Çıktı video: H.264/mp4 (tarayıcı uyumlu)
                    """)

                # Sağ: durum
                with gr.Column(scale=1):
                    gr.Markdown("""
                    ### ▶️ Nasıl Kullanılır?
                    1. Sol üstten video yükle
                    2. Ayarları düzenle (hız/kalite dengesi)
                    3. **Videoyu İşle** butonuna bas
                    4. Colab terminalinden ilerlemeyi takip et
                    5. İki annotated video aşağıda belirecek

                    ### 🧪 Test Önerileri
                    - **Kalabalık sahne:** yüksek yoğunluklu kampüs geçişi
                    - **Hareket bulanıklığı:** hızlı yürüyen/bisikletli
                    - **Uzak mesafe:** uzaktan çekilmiş geniş açı
                    """)

            gr.Markdown("---")
            gr.Markdown("### 📹 Sonuç Videoları")
            with gr.Row():
                video_yolo_out  = gr.Video(label="⚡ YOLOv8n Sonucu",        height=340)
                video_frcnn_out = gr.Video(label="🎯 Faster R-CNN Sonucu",   height=340)
            video_metrics = gr.DataFrame(label="📊 Video İstatistikleri", wrap=True)

            video_btn.click(
                fn=process_video,
                inputs=[video_input, v_conf, v_iou, v_skip, v_maxs],
                outputs=[video_yolo_out, video_frcnn_out, video_metrics]
            )

        # ────────────────────────────────────────────────────────
        # SEKME 3: Failure Case Analizi
        # ────────────────────────────────────────────────────────
        with gr.Tab("❌ Failure Case Analizi"):
            gr.Markdown("""
            ## Başarısız Durum Analizi
            Modellerin zorlandığı senaryoları incelemek için kullanın.
            Düşük güven eşiği, hem başarıları hem de yanlış pozitif (FP) örnekleri ortaya çıkarır.
            """)
            with gr.Row():
                fail_img  = gr.Image(label="Görüntü Yükle (oklüzyon, gece, uzak mesafe vb.)",
                                     type="numpy", sources=["upload", "webcam"])
                fail_conf = gr.Slider(0.10, 0.90, value=0.25, step=0.05,
                                      label="Güven Eşiği (FP görmek için düşük tutun: 0.20–0.35)")
            fail_btn = gr.Button("🔍 Failure Case Analiz Et", variant="secondary")
            with gr.Row():
                fail_yolo_out  = gr.Image(label="YOLOv8n Sonucu")
                fail_frcnn_out = gr.Image(label="Faster R-CNN Sonucu")
            fail_metrics = gr.DataFrame(label="Metrikler")

            gr.Markdown("""
            ---
            ### 📋 Referans: Bilinen Failure Case Kategorileri

            | # | Senaryo | YOLOv8n Davranışı | Faster R-CNN Davranışı | Teknik Neden |
            |---|---------|-------------------|------------------------|--------------|
            | FC-1 | Oklüzyon (üst üste kişiler) | FN — arkadaki kaçırılır | Biraz daha iyi ama NMS bastırır | Yüksek IoU → NMS overlap bastırması |
            | FC-2 | Küçük nesne (uzak mesafe) | Kaçırır | Kaçırır | Stride-32'de özellik haritası çok küçük |
            | FC-3 | Hareket bulanıklığı | FP artışı | Bbox genişler | COCO eğitim seti net görüntü ağırlıklı |
            | FC-4 | Gece / düşük ışık | Recall düşer | Recall düşer | Normalizasyon dağılım kayması (domain shift) |
            | FC-5 | Benzer renk arka plan | FP (duvarı kişi sanır) | RPN hatalı proposal | Renk/doku özelliklerine aşırı bağımlılık |
            """)

            fail_btn.click(
                fn=lambda img, conf: compare_models(img, conf, 0.45),
                inputs=[fail_img, fail_conf],
                outputs=[fail_yolo_out, fail_frcnn_out, fail_metrics]
            )

        # ────────────────────────────────────────────────────────
        # SEKME 4: FPS Benchmark
        # ────────────────────────────────────────────────────────
        with gr.Tab("⚡ FPS Benchmark"):
            gr.Markdown("""
            ## Hız Karşılaştırması
            Aynı görüntüyü 20 kez çalıştırarak ortalama / std sapma / min / max FPS ölçer.
            İlk 2 çalıştırma 'warmup' olarak atılır.
            """)
            bench_img = gr.Image(label="Benchmark Görüntüsü", type="numpy", height=280)
            bench_btn = gr.Button("🏁 Benchmark Başlat (20 çalıştırma)", variant="secondary")
            bench_out = gr.DataFrame(label="📊 Benchmark Sonuçları")

            def run_bench(img):
                if img is None:
                    return pd.DataFrame({"Uyarı": ["Görüntü yükleyin."]})
                return fps_benchmark(img, n_runs=20)

            bench_btn.click(fn=run_bench, inputs=[bench_img], outputs=[bench_out])

        # ────────────────────────────────────────────────────────
        # SEKME 5: Model Bilgisi
        # ────────────────────────────────────────────────────────
        with gr.Tab("ℹ️ Model Bilgisi"):
            gr.Markdown(f"""
            ## Yüklü Modeller ve Teknik Detaylar

            ### ⚡ YOLOv8n — You Only Look Once (Nano)
            | Özellik | Değer |
            |---------|-------|
            | Mimari | Single-stage, anchor-free, CSPNet backbone |
            | Parametre | ~{sum(p.numel() for p in yolo_model.model.parameters())/1e6:.1f}M |
            | Sınıf | 80 (COCO) |
            | Çıktı | Tek geçişte bbox + sınıf olasılığı |
            | Avantaj | Hız (gerçek zamanlı işlem), küçük model boyutu |
            | Dezavantaj | Küçük / üst üste binen nesnelerde kayıp |

            ### 🎯 Faster R-CNN — ResNet-50-FPN
            | Özellik | Değer |
            |---------|-------|
            | Mimari | Two-stage: RPN + ROI Head, anchor-based |
            | Parametre | ~{sum(p.numel() for p in frcnn_model.parameters())/1e6:.1f}M |
            | Sınıf | 80 (COCO) |
            | Çıktı | Stage-1: proposal → Stage-2: sınıflandırma |
            | Avantaj | Yüksek doğruluk, oklüzyona dayanıklılık |
            | Dezavantaj | Yavaş (iki aşamalı), gerçek zamanlı zor |

            ### 🖥️ Ortam Bilgisi
            | | |
            |--|--|
            | Cihaz | {DEVICE} |
            | GPU | {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'Yok'} |
            | PyTorch | {torch.__version__} |
            | Torchvision | {torchvision.__version__} |
            | Pre-training | COCO 2017 (118K görüntü, 80 sınıf) |
            """)

    print("\n🚀 Gradio arayüzü başlatılıyor...")
    print("⏳ 'Running on public URL' satırını bekleyin — linke tıklayın")
    demo.launch(share=True, debug=False, show_error=True)


In [ ]:
# ============================================================
# HÜCRE 9 — DEMO'YU BAŞLAT
# ============================================================
build_and_launch()


🚀 Gradio arayüzü başlatılıyor...
⏳ 'Running on public URL' satırını bekleyin — linke tıklayın
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73f54f69c0e7809127.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
